# Manga Colorization (GPU T4 x2)

Generated by Manga Color Studio. The worker code is `scripts/colorize_worker.py` in the app;
this run's settings are in its `CONFIG` line.


In [ ]:
import hashlib, os
_key = ''
for _root, _dirs, _files in os.walk('/kaggle/input'):
    if 'mcs-key.txt' in _files:
        with open(os.path.join(_root, 'mcs-key.txt'), encoding='utf-8') as _f:
            _key = _f.read().strip()
        break
if not _key:
    raise RuntimeError('Encryption is on, but the temporary key dataset is not mounted in this notebook. '
                       'Start the run again from the app.')
_fp = hashlib.sha256(b'mcs-fingerprint:' + _key.encode('utf-8')).hexdigest()[:12]
if _fp != '572149899184':
    raise RuntimeError('The mounted encryption key is not the key set in the app (fingerprints %s vs %s). '
                       'Start the run again from the app.' % (_fp, '572149899184'))
del _key
print('Encryption key verified (fingerprint %s).' % _fp)


In [ ]:
!pip uninstall -y -q google-tunix 2>/dev/null; pip install -q --upgrade "transformers==5.15.0" "tokenizers==0.22.2" accelerate safetensors peft kaggle cryptography
!pip install --upgrade "huggingface_hub @ git+https://github.com/huggingface/huggingface_hub.git@f34cd32cac1f3603e3294543e1fb0eb4508d5139" "diffusers @ git+https://github.com/huggingface/diffusers.git@102c24934fe1705e6caaadb03789b0f2542fd8a9"


In [ ]:
import os

import huggingface_hub
try:
    from huggingface_hub import get_cached_repo_tree  # noqa: F401
except ImportError:
    raise RuntimeError('Installed huggingface_hub is too old for diffusers main (' + huggingface_hub.__version__
                       + '); re-run the install cell or restart the session and Run All.')
import torch
import diffusers
import transformers

print('huggingface_hub:', huggingface_hub.__version__)
print('torch:', torch.__version__, '| diffusers:', diffusers.__version__, '| transformers:', transformers.__version__)
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU in this session. Check the notebook accelerator setting.')
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print('GPU %d: %s, %.1f GB, compute %d.%d' % (i, p.name, p.total_memory / 1e9, p.major, p.minor))


In [ ]:
import os, sys
WORKER_SOURCE = 'import json\nCONFIG = json.loads(\'{"backend": "gpu", "model_repo": "circlestone-labs/Anima-Base-v1.0-Diffusers", "model_revision": "main", "loras": [{"name": "Color Manga Anima (Base Style)", "path": "colorMangaAnima.safetensors", "scale": 1.0}, {"name": "Anima Turbo (8-Step Fast)", "path": "anima-turbo-lora-v0.2.safetensors", "scale": 1.0}], "max_dim": 1024, "num_steps": 8, "guidance_scale": 1.0, "prompt": "masterpiece, best quality, anime style, highly detailed coloring, vibrant color harmony, crisp lineart", "negative_prompt": "worst quality, low quality, blurry, artifact, oversaturated, deformed, greyscale, monochrome", "encrypted": true, "key_fingerprint": "572149899184"}\')\n\n"""Colorization worker that runs inside the Kaggle notebook, on TPU or GPU.\n\nscripts/make_notebook.py writes this file into the notebook with `CONFIG = {...}` prepended\n(the run\'s settings: backend, model, LoRAs, prompt...). The notebook calls `main()`, which:\n\n1. finds the cached model / LoRAs in the mounted datasets (or downloads them),\n2. collects the scans (splitting double-page spreads on TPU, see below),\n3. starts one process per TPU core or GPU, each colorizing its share of the pages,\n4. stitches spreads back together and zips the results for download.\n\nBackends:\n- "tpu": 8 TPU cores via torch_xla, bfloat16. Every new image shape costs a ~4 minute XLA\n  compile, so all pages share one fixed portrait canvas and spreads are split in two.\n- "gpu": one process per CUDA GPU (2 on Kaggle\'s T4 x2). T4/P100 have no bfloat16, so the\n  pipeline runs in float16 and is rebuilt in float32 if a page comes out blank (overflow).\n  Any shape is cheap, so each page keeps its own aspect ratio and spreads stay whole.\n- "cpu": local testing only.\n\nNothing secret is embedded: an optional HF_TOKEN is read from Kaggle Secrets.\n\nEncryption (CONFIG["encrypted"]): the scans arrive as an AES-encrypted .mcsenc archive and the\nresults leave as one. The key comes from the MCS_ENCRYPTION_KEY Kaggle Secret and is checked\nagainst the app\'s key fingerprint. Every readable page lives only in /tmp, never in\n/kaggle/working (which Kaggle saves as the notebook\'s output), so even a crash mid-run leaves\nnothing readable behind.\n"""\nimport gc\nimport json\nimport math\nimport os\nimport re\nimport shutil\nimport subprocess\nimport sys\nimport time\nimport traceback\nimport urllib.request\nimport zipfile\nfrom pathlib import Path\n\n# Must be set before CUDA initialises: lets PyTorch grow memory segments instead of failing\n# on fragmentation when a T4 is nearly full.\nos.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")\n\nimport numpy as np\nimport torch\nfrom PIL import Image\n\nCONFIG = globals().get("CONFIG") or {}\n\nIMAGE_EXTS = (".jpg", ".jpeg", ".png", ".webp", ".bmp")\nCOMPONENTS = ["text_encoder", "tokenizer", "t5_tokenizer", "text_conditioner", "transformer", "vae"]\nT0 = time.time()\n\n\ndef cfg(key, default=None):\n    return CONFIG.get(key, default)\n\n\ndef input_root():\n    return cfg("input_root", "/kaggle/input")\n\n\ndef work_dir():\n    return cfg("work_dir", "/kaggle/working")\n\n\ndef encrypted():\n    return bool(cfg("encrypted"))\n\n\ndef pages_dir():\n    """Where scans and colorized pages live during the run. When encrypting, a folder outside\n    /kaggle/working so nothing readable can end up in the saved notebook output."""\n    return cfg("scratch_dir", "/tmp/mcs_work") if encrypted() else work_dir()\n\n\n_KEY = {"value": None}\n\n\ndef load_key():\n    """The passphrase from the app\'s temporary key dataset, checked against the app\'s fingerprint."""\n    import mcs_crypto\n    key = ""\n    for root, _dirs, files in os.walk(input_root()):\n        if mcs_crypto.KEY_FILE in files:\n            with open(os.path.join(root, mcs_crypto.KEY_FILE), encoding="utf-8") as f:\n                key = f.read().strip()\n            break\n    if not key:\n        raise RuntimeError(\n            "Encryption is on, but the temporary key dataset is not mounted in this notebook. "\n            "Start the run again from the app.")\n    if mcs_crypto.fingerprint(key) != cfg("key_fingerprint"):\n        raise RuntimeError(\n            "The mounted encryption key is not the key set in the app (fingerprints %s vs %s). "\n            "Start the run again from the app." % (mcs_crypto.fingerprint(key), cfg("key_fingerprint")))\n    _KEY["value"] = key\n    return key\n\n\ndef local_model_dir():\n    return cfg("local_model_dir", "/tmp/mcs_model")\n\n\ndef mark(msg):\n    line = "[%7.1fs] %s" % (time.time() - T0, msg)\n    print(line, flush=True)\n    try:\n        with open(os.path.join(work_dir(), "debug.log"), "a", encoding="utf-8") as f:\n            f.write(line + "\\n")\n    except OSError:\n        pass\n\n\n# ------------------------------------------------------------------ backends\n\ndef backend():\n    return cfg("backend", "tpu")\n\n\ndef fixed_canvas():\n    return backend() == "tpu"\n\n\ndef get_device(rank):\n    if backend() == "tpu":\n        import torch_xla.core.xla_model as xm\n        return xm.xla_device()\n    if backend() == "gpu":\n        torch.cuda.set_device(rank)\n        return torch.device("cuda", rank)\n    return torch.device("cpu")\n\n\ndef sync(wait=True):\n    """Flushes queued device work (XLA graphs) / waits for CUDA kernels."""\n    if backend() == "tpu":\n        import torch_xla\n        torch_xla.sync(wait=wait)\n    elif backend() == "gpu" and wait:\n        torch.cuda.synchronize()\n\n\ndef mark_step():\n    if backend() == "tpu":\n        import torch_xla.core.xla_model as xm\n        xm.mark_step()\n\n\ndef mem_info(device):\n    try:\n        if backend() == "tpu":\n            import torch_xla.core.xla_model as xm\n            info = xm.get_memory_info(device)\n            return "%.1f/%.1f GB" % (info["bytes_used"] / 1e9, info["bytes_limit"] / 1e9)\n        if backend() == "gpu":\n            return "%.1f/%.1f GB" % (torch.cuda.memory_allocated(device) / 1e9,\n                                     torch.cuda.get_device_properties(device).total_memory / 1e9)\n    except Exception:\n        pass\n    return "n/a"\n\n\ndef precision_plans():\n    """Ordered {component: dtype} plans. A worker starts with the first and moves to the next\n    if a page comes out blank (float16 overflow) or a plan fails outright.\n\n    GPU (T4/P100 have no native bfloat16):\n      1. transformer bfloat16 (emulated, but float32\'s range), the rest float32: ~7.4 GB.\n         The Anima transformer overflows float16 on a T4 (NaN pages, seen on Kaggle), so\n         bfloat16 goes first.\n      2. transformer float16, the rest float32: faster where it doesn\'t overflow.\n      3. everything float32: ~11.5 GB, tight on a 15 GB T4 even with VAE tiling.\n    TPU: bfloat16 throughout. CPU (tests): cpu_dtype throughout.\n    """\n    def plan(transformer, rest):\n        return {c: (transformer if c == "transformer" else rest) for c in COMPONENTS}\n\n    if backend() == "gpu":\n        return [plan(torch.bfloat16, torch.float32), plan(torch.float16, torch.float32),\n                plan(torch.float32, torch.float32)]\n    dt = getattr(torch, cfg("cpu_dtype", "bfloat16")) if backend() == "cpu" else torch.bfloat16\n    return [plan(dt, dt)]\n\n\ndef describe_plan(plan):\n    t, rest = plan["transformer"], plan["text_encoder"]\n    name = lambda d: str(d).replace("torch.", "")\n    return name(t) if t == rest else "transformer %s, rest %s" % (name(t), name(rest))\n\n\ndef world_size():\n    if backend() == "tpu":\n        return 8\n    if backend() == "gpu":\n        return max(1, torch.cuda.device_count())\n    return 1\n\n\n# ------------------------------------------------------------------ model & LoRA files\n\ndef secret(name):\n    try:\n        from kaggle_secrets import UserSecretsClient\n        return UserSecretsClient().get_secret(name) or ""\n    except Exception:\n        return ""\n\n\ndef prepare_local_model():\n    """Finds the mounted model dataset and makes it loadable without network access.\n\n    modular_model_index.json names each component\'s source repo, so the dataset copy would\n    still be fetched from Hugging Face. A rewritten index points every component at the\n    mounted folder instead.\n    """\n    root_dir = input_root()\n    if not os.path.exists(root_dir):\n        return None\n    for root, dirs, files in os.walk(root_dir):\n        if "modular_model_index.json" in files:\n            with open(os.path.join(root, "modular_model_index.json")) as f:\n                idx = json.load(f)\n            for spec in idx.values():\n                if isinstance(spec, list) and len(spec) == 3 and isinstance(spec[2], dict) and "pretrained_model_name_or_path" in spec[2]:\n                    spec[2]["pretrained_model_name_or_path"] = root\n                    spec[2]["revision"] = None\n            os.makedirs(local_model_dir(), exist_ok=True)\n            with open(os.path.join(local_model_dir(), "modular_model_index.json"), "w") as f:\n                json.dump(idx, f, indent=2)\n            return local_model_dir()\n        if "model_index.json" in files:\n            return root\n    return None\n\n\ndef find_local_model():\n    """The model folder prepared by prepare_local_model() in the main process, if any."""\n    if os.path.isfile(os.path.join(local_model_dir(), "modular_model_index.json")):\n        return local_model_dir()\n    if os.path.exists(input_root()):\n        for root, dirs, files in os.walk(input_root()):\n            if "model_index.json" in files:\n                return root\n    return None\n\n\ndef _model_repo():\n    return cfg("model_repo", "circlestone-labs/Anima-Base-v1.0-Diffusers")\n\n\ndef _model_revision():\n    return cfg("model_revision") or "main"\n\n\ndef _download_with_watchdog(repo, filename, size, revision):\n    """hf_hub_download in a subprocess that is killed if it stalls (HF CDN hiccups)."""\n    code = ("from huggingface_hub import hf_hub_download; "\n            "p = hf_hub_download(%r, %r, revision=%r); print(\'OK\', p)" % (repo, filename, revision))\n    timeout = 300 if size < 50 * 1024 * 1024 else max(600, int(size / (500 * 1024)))\n    cache = Path(os.environ.get("HF_HOME", os.path.expanduser("~/.cache/huggingface")))\n\n    def cache_bytes():\n        total = 0\n        for d in (cache / "hub", cache / "xet"):\n            if d.is_dir():\n                for p in d.rglob("*"):\n                    try:\n                        total += p.stat().st_size if p.is_file() else 0\n                    except OSError:\n                        pass\n        return total\n\n    t0 = last_progress = time.time()\n    prev = cache_bytes()\n    proc = subprocess.Popen([sys.executable, "-c", code], stdout=subprocess.PIPE, stderr=subprocess.PIPE)\n    while proc.poll() is None:\n        time.sleep(5)\n        cur = cache_bytes()\n        if cur > prev:\n            prev, last_progress = cur, time.time()\n        elif time.time() - last_progress > 180:\n            proc.kill()\n            mark("download of %s stalled for 3 minutes, retrying" % filename)\n            return False\n        if time.time() - t0 > timeout:\n            proc.kill()\n            mark("download of %s timed out after %ds, retrying" % (filename, timeout))\n            return False\n    out, err = proc.communicate()\n    if proc.returncode != 0 or b"OK" not in out:\n        mark("download of %s failed: %s" % (filename, err.decode(errors="replace")[-300:]))\n        return False\n    return True\n\n\ndef warm_model_cache():\n    """Pre-downloads the model from Hugging Face when no cached dataset is mounted."""\n    repo, revision = _model_repo(), _model_revision()\n    url = "https://huggingface.co/api/models/%s/revision/%s?blobs=true" % (repo, revision)\n    with urllib.request.urlopen(url, timeout=30) as r:\n        data = json.loads(r.read().decode())\n    skip = {"README.md", "LICENSE.md", ".gitattributes"}\n    files = {s["rfilename"]: (s.get("size") or 0) for s in data.get("siblings", []) if s["rfilename"] not in skip}\n    for name in sorted(files, key=lambda n: files[n]):  # small config files first\n        for attempt in range(1, 5):\n            mark("downloading %s (attempt %d/4)..." % (name, attempt))\n            if _download_with_watchdog(repo, name, files[name], revision):\n                break\n            time.sleep(2 * attempt)\n\n\ndef resolve_lora_path(lora_path_str):\n    if os.path.isfile(lora_path_str):\n        return lora_path_str\n    filename = os.path.basename(lora_path_str).split("?")[0]\n\n    # 1. Mounted Kaggle datasets & files downloaded earlier in this session\n    if os.path.exists(input_root()):\n        for root, dirs, files in os.walk(input_root()):\n            if filename in files:\n                return os.path.join(root, filename)\n    lora_dir = os.path.join(work_dir(), "loras")\n    if os.path.isfile(os.path.join(lora_dir, filename)):\n        return os.path.join(lora_dir, filename)\n\n    # 2. Direct URL\n    if lora_path_str.startswith(("http://", "https://")):\n        os.makedirs(lora_dir, exist_ok=True)\n        dest = os.path.join(lora_dir, filename)\n        mark("Downloading LoRA from URL: %s" % lora_path_str)\n        headers = {"User-Agent": "Mozilla/5.0"}\n        if os.getenv("HF_TOKEN") and "huggingface.co" in lora_path_str.split("/")[2]:\n            headers["Authorization"] = "Bearer " + os.environ["HF_TOKEN"]\n        req = urllib.request.Request(lora_path_str, headers=headers)\n        with urllib.request.urlopen(req, timeout=300) as resp, open(dest, "wb") as out_f:\n            shutil.copyfileobj(resp, out_f)\n        return dest\n\n    # 3. Hugging Face path: org/repo/file.safetensors\n    if lora_path_str.count("/") >= 2:\n        from huggingface_hub import hf_hub_download\n        parts = lora_path_str.split("/")\n        return hf_hub_download(repo_id="/".join(parts[:2]), filename="/".join(parts[2:]))\n\n    raise FileNotFoundError("LoRA %r is not in any mounted dataset and has no download source." % lora_path_str)\n\n\n# ------------------------------------------------------------------ LoRA fusion\n\n_KOHYA_TO_DIFFUSERS = {\n    "self_attn_q_proj": "attn1.to_q",\n    "self_attn_k_proj": "attn1.to_k",\n    "self_attn_v_proj": "attn1.to_v",\n    "self_attn_output_proj": "attn1.to_out.0",\n    "cross_attn_q_proj": "attn2.to_q",\n    "cross_attn_k_proj": "attn2.to_k",\n    "cross_attn_v_proj": "attn2.to_v",\n    "cross_attn_output_proj": "attn2.to_out.0",\n    "mlp_layer1": "ff.net.0.proj",\n    "mlp_layer2": "ff.net.2",\n}\n\n_TURBO_MAP = {\n    "self_attn.q_proj": "attn1.to_q",\n    "self_attn.k_proj": "attn1.to_k",\n    "self_attn.v_proj": "attn1.to_v",\n    "self_attn.output_proj": "attn1.to_out.0",\n    "cross_attn.q_proj": "attn2.to_q",\n    "cross_attn.k_proj": "attn2.to_k",\n    "cross_attn.v_proj": "attn2.to_v",\n    "cross_attn.output_proj": "attn2.to_out.0",\n    "mlp.layer1": "ff.net.0.proj",\n    "mlp.layer2": "ff.net.2",\n}\n\n\ndef lora_factors(sd):\n    """{transformer weight name: (A, B, alpha)} for a Kohya or diffusion_model-style LoRA."""\n    weights_A, weights_B, alphas = {}, {}, {}\n    if any(k.startswith("lora_unet_blocks_") for k in sd.keys()):\n        for key, value in sd.items():\n            if not key.startswith("lora_unet_blocks_"):\n                continue\n            parts = key.split(".")\n            tokens = parts[0][len("lora_unet_blocks_"):].split("_")\n            diffusers_submod = _KOHYA_TO_DIFFUSERS.get("_".join(tokens[1:]))\n            if diffusers_submod is None:\n                continue\n            target_name = "transformer_blocks.%s.%s.weight" % (tokens[0], diffusers_submod)\n            rest = parts[1:]\n            if rest == ["alpha"]:\n                alphas[target_name] = value.item() if hasattr(value, "item") else float(value)\n            elif len(rest) == 2 and rest[1] == "weight":\n                if rest[0] == "lora_down":\n                    weights_A[target_name] = value\n                elif rest[0] == "lora_up":\n                    weights_B[target_name] = value\n    else:\n        for k, v in sd.items():\n            m = re.match(r"diffusion_model\\.blocks\\.(\\d+)\\.(.*?)\\.(lora_[AB]\\.weight|alpha)", k)\n            if not m:\n                continue\n            block_idx, submod, param_type = m.groups()\n            diff_sub = _TURBO_MAP.get(submod)\n            if not diff_sub:\n                continue\n            target_name = "transformer_blocks.%s.%s.weight" % (block_idx, diff_sub)\n            if param_type == "alpha":\n                alphas[target_name] = v.item() if hasattr(v, "item") else float(v)\n            elif "lora_A" in param_type:\n                weights_A[target_name] = v\n            else:\n                weights_B[target_name] = v\n\n    return {name: (A, weights_B[name], alphas.get(name, float(A.shape[0])))\n            for name, A in weights_A.items() if name in weights_B}\n\n\ndef add_lora(lora_path, scale, transformer, tag=""):\n    """Adds a LoRA\'s weight deltas straight into the transformer weights (no PEFT overhead).\n\n    Called while the model is still on the CPU: thousands of small in-place adds are near\n    instant there, whereas on TPU each one is a separate XLA graph (~5 minutes in total).\n    Each full-size delta (B @ A) is built and applied one at a time: all of them at once\n    would be ~7.5 GB of float32 per process, enough to exhaust a GPU machine\'s RAM.\n    """\n    from safetensors.torch import load_file\n    file_path = resolve_lora_path(str(lora_path))\n    factors = lora_factors(load_file(file_path))\n    params = dict(transformer.named_parameters())\n    fused = 0\n    with torch.no_grad():\n        for name, (A, B, alpha) in factors.items():\n            param = params.get(name)\n            if param is None:\n                continue\n            A, B = A.float(), B.float()\n            delta = (B @ A) * ((alpha / A.shape[0]) * scale)\n            if param.shape == delta.shape:\n                param.add_(delta.to(device=param.device, dtype=param.dtype))\n                fused += 1\n            del delta\n    mark("%s Fused LoRA %s (scale %.2f): %d matrices" % (tag, os.path.basename(file_path), scale, fused))\n    return fused\n\n\n# ------------------------------------------------------------------ pipeline\n\ndef install_reference_hook(pipe):\n    """Feeds the encoded B&W page to the transformer as an extra frame, so the model colors\n    the existing lineart instead of drawing a new image."""\n    orig_forward = pipe.transformer.forward\n\n    def reference_forward(hidden_states, timestep, encoder_hidden_states=None, **kwargs):\n        ref_latent = getattr(pipe.transformer, "_ref_latent", None)\n        if ref_latent is None:\n            return orig_forward(hidden_states=hidden_states, timestep=timestep, encoder_hidden_states=encoder_hidden_states, **kwargs)\n        is_4d = hidden_states.ndim == 4\n        if is_4d:\n            hidden_states = hidden_states.unsqueeze(2)\n        out = orig_forward(hidden_states=torch.cat([hidden_states, ref_latent], dim=2), timestep=timestep,\n                           encoder_hidden_states=encoder_hidden_states, **kwargs)\n        if hasattr(out, "sample"):\n            sample = out.sample[:, :, :1, :, :]\n            out.sample = sample.squeeze(2) if is_4d else sample\n            return out\n        sample = out[0][:, :, :1, :, :]\n        return (sample.squeeze(2) if is_4d else sample,)\n\n    pipe.transformer.forward = reference_forward\n\n\ndef install_step_sync(pipe):\n    """Cuts the XLA graph after every denoising step (no-op on GPU/CPU)."""\n    sched = getattr(pipe, "scheduler", None)\n    if sched is None or not hasattr(sched, "step"):\n        return\n    orig_step = sched.step\n\n    def step(*args, **kwargs):\n        out = orig_step(*args, **kwargs)\n        mark_step()\n        return out\n\n    sched.step = step\n\n\ndef _load_components(pipe, plan, max_attempts=3):\n    groups = {}\n    for name, dt in plan.items():\n        groups.setdefault(dt, []).append(name)\n    for _ in range(max_attempts):\n        for dt, names in groups.items():\n            missing = [n for n in names if getattr(pipe, n, None) is None]\n            if missing:\n                pipe.load_components(names=missing, dtype=dt)\n        # Loading by name skips components without weights, so ask for the scheduler explicitly.\n        if getattr(pipe, "scheduler", None) is None:\n            pipe.load_components(names=["scheduler"])\n        if all(getattr(pipe, c, None) is not None for c in COMPONENTS + ["scheduler"]):\n            return\n        time.sleep(10)\n    raise RuntimeError("Failed to load components %r" % [c for c in COMPONENTS + ["scheduler"] if getattr(pipe, c, None) is None])\n\n\ndef configure_guidance(pipe):\n    """The Anima pipeline takes its CFG scale from its `guider` component (default 4.0), not\n    from a call argument. At 1.0 the guider skips the unconditional pass, which halves the\n    work per step and is what the Turbo LoRA is distilled for."""\n    from diffusers.guiders import ClassifierFreeGuidance\n    pipe.update_components(guider=ClassifierFreeGuidance(guidance_scale=float(cfg("guidance_scale", 1.0))))\n\n\ndef load_pipeline(plan, tag):\n    from diffusers import ModularPipeline\n    local = find_local_model()\n    if local:\n        try:\n            pipe = ModularPipeline.from_pretrained(local)\n            _load_components(pipe, plan)\n            mark("%s Loaded model from mounted dataset (%s)" % (tag, describe_plan(plan)))\n            return pipe\n        except Exception as e:\n            mark("%s Mounted model failed to load (%r); falling back to %s" % (tag, e, _model_repo()))\n    pipe = ModularPipeline.from_pretrained(_model_repo(), revision=_model_revision())\n    _load_components(pipe, plan)\n    mark("%s Loaded model from Hugging Face (%s)" % (tag, describe_plan(plan)))\n    return pipe\n\n\ndef build_pipeline(device, plan, tag):\n    pipe = load_pipeline(plan, tag)\n    configure_guidance(pipe)\n    if backend() == "gpu":\n        # Decode/encode in overlapping tiles: the full-page VAE pass is the memory peak on a T4.\n        pipe.vae.enable_tiling()\n    for lora in cfg("loras", []):\n        add_lora(lora["path"], float(lora.get("scale", 1.0)), pipe.transformer, tag)\n    pipe = pipe.to(device)\n    sync(wait=True)\n    install_reference_hook(pipe)\n    install_step_sync(pipe)\n    return pipe\n\n\n# ------------------------------------------------------------------ pages\n\ndef canvas_size(w, h, max_dim):\n    """Generation size for a page of w x h.\n\n    TPU: one fixed portrait canvas for every page (any new shape triggers a slow XLA compile).\n    GPU/CPU: the page\'s own aspect ratio at the same pixel budget, sides multiples of 32.\n    """\n    canvas_h = max(32, int(round(max_dim / 32.0)) * 32)\n    canvas_w = max(32, int(round(max_dim * (1024.0 / 1440.0) / 32.0)) * 32)\n    if fixed_canvas():\n        return canvas_w, canvas_h\n    s = math.sqrt(canvas_w * canvas_h / float(w * h))\n    return max(32, int(w * s) // 32 * 32), max(32, int(h * s) // 32 * 32)\n\n\ndef prepare_canvas(img, max_dim):\n    """Returns (canvas, crop_box): the page scaled (and on TPU letterboxed) for generation."""\n    orig_w, orig_h = img.size\n    canvas_w, canvas_h = canvas_size(orig_w, orig_h, max_dim)\n    if not fixed_canvas():\n        return img.resize((canvas_w, canvas_h), Image.LANCZOS), (0, 0, canvas_w, canvas_h)\n    scale = min(canvas_w / orig_w, canvas_h / orig_h)\n    scaled_w = max(1, min(canvas_w, int(round(orig_w * scale))))\n    scaled_h = max(1, min(canvas_h, int(round(orig_h * scale))))\n    canvas = Image.new("RGB", (canvas_w, canvas_h), (255, 255, 255))\n    pad_x, pad_y = (canvas_w - scaled_w) // 2, (canvas_h - scaled_h) // 2\n    canvas.paste(img.resize((scaled_w, scaled_h), Image.LANCZOS), (pad_x, pad_y))\n    return canvas, (pad_x, pad_y, pad_x + scaled_w, pad_y + scaled_h)\n\n\ndef encode_reference(pipe, canvas, device):\n    arr = np.array(canvas).astype(np.float32) / 127.5 - 1.0\n    ref = torch.from_numpy(arr).permute(2, 0, 1).unsqueeze(0).unsqueeze(2).to(device=device, dtype=pipe.vae.dtype)\n    with torch.no_grad():\n        enc = pipe.vae.encode(ref)\n        if hasattr(enc, "latent_dist"):\n            latent = enc.latent_dist.sample()\n        elif isinstance(enc, tuple):\n            latent = enc[0]\n        else:\n            latent = getattr(enc, "latents", enc)\n        if latent.ndim == 4:\n            latent = latent.unsqueeze(2)\n        vcfg = pipe.vae.config\n        if getattr(vcfg, "latents_mean", None) is not None:\n            mean = torch.tensor(vcfg.latents_mean, device=latent.device, dtype=latent.dtype).view(1, -1, 1, 1, 1)\n            std = torch.tensor(vcfg.latents_std, device=latent.device, dtype=latent.dtype).view(1, -1, 1, 1, 1)\n            latent = (latent - mean) / std\n        elif getattr(vcfg, "scaling_factor", None) is not None:\n            latent = latent * vcfg.scaling_factor\n    # joined with the transformer\'s own latents in the reference hook\n    return latent.to(pipe.transformer.dtype)\n\n\ndef looks_broken(img):\n    """float16 overflow shows up as a flat (usually black) image."""\n    return float(np.asarray(img.convert("L"), dtype=np.float32).std()) < 2.0\n\n\ndef colorize_page(pipe, path, device):\n    orig = Image.open(path).convert("RGB")\n    canvas, crop_box = prepare_canvas(orig, int(cfg("max_dim", 1440)))\n    w, h = canvas.size\n    pipe.transformer._ref_latent = encode_reference(pipe, canvas, device)\n    mark_step()\n    kwargs = {\n        "prompt": [cfg("prompt", "")],\n        "num_inference_steps": int(cfg("num_steps", 8)),\n        "height": h,\n        "width": w,\n    }\n    if cfg("negative_prompt") and float(cfg("guidance_scale", 1.0)) > 1.0:\n        kwargs["negative_prompt"] = [cfg("negative_prompt")]\n    try:\n        result = pipe(**kwargs)\n        mark_step()\n        out = result.images[0].crop(crop_box)\n    finally:\n        pipe.transformer._ref_latent = None\n    return out.resize(orig.size, Image.LANCZOS) if out.size != orig.size else out\n\n\ndef release_memory():\n    """Frees a pipeline the caller has already dropped all references to. The reference hook\'s\n    closure makes a cycle (pipe -> transformer.forward -> pipe), so only a full gc frees it."""\n    gc.collect()\n    if backend() == "gpu":\n        torch.cuda.empty_cache()\n\n\ndef mp_worker(rank, image_paths, n_workers):\n    tag = "[%s %d/%d]" % ("Core" if backend() == "tpu" else backend().upper(), rank, n_workers)\n    device = get_device(rank)\n    plans = precision_plans()\n    plan_idx = 0\n    mark("%s Initializing pipeline on %s..." % (tag, device))\n    pipe = build_pipeline(device, plans[plan_idx], tag)\n\n    out_dir = os.path.join(pages_dir(), "generated")\n    os.makedirs(out_dir, exist_ok=True)\n    my_paths = image_paths[rank::n_workers]\n    mark("%s Ready! Assigned %d images | memory: %s" % (tag, len(my_paths), mem_info(device)))\n\n    t_start = time.time()\n    for i, path in enumerate(my_paths):\n        name = os.path.basename(path)\n        out_path = os.path.join(out_dir, "edit_%s.png" % os.path.splitext(name)[0])\n        if os.path.exists(out_path):\n            continue\n        t1 = time.time()\n        mark("%s [%d/%d] %s -> generating..." % (tag, i + 1, len(my_paths), name))\n        while True:\n            try:\n                colorized = colorize_page(pipe, path, device)\n                problem = "a blank page (overflow)" if looks_broken(colorized) else None\n            except Exception as e:\n                traceback.print_exc()\n                colorized, problem = None, repr(e)\n            if problem is None:\n                colorized.save(out_path)\n                mark("%s [%d/%d] DONE %s (%ds, %s, mem %s)" % (tag, i + 1, len(my_paths), name, time.time() - t1,\n                                                               describe_plan(plans[plan_idx]), mem_info(device)))\n                break\n            if plan_idx + 1 >= len(plans):\n                mark("%s [%d/%d] FAILED %s: %s" % (tag, i + 1, len(my_paths), name, problem))\n                try:\n                    sync(wait=False)\n                except Exception:\n                    pass\n                break\n            # Switch precision for this page and every later one.\n            plan_idx += 1\n            mark("%s %s gave %s; switching to %s" % (tag, describe_plan(plans[plan_idx - 1]), problem,\n                                                     describe_plan(plans[plan_idx])))\n            pipe = colorized = None  # drop every reference first, then collect\n            release_memory()\n            try:\n                pipe = build_pipeline(device, plans[plan_idx], tag)\n            except Exception as e:\n                traceback.print_exc()\n                mark("%s could not build %s: %r" % (tag, describe_plan(plans[plan_idx]), e))\n                if plan_idx + 1 >= len(plans):\n                    mark("%s no precision left to try; stopping this worker" % tag)\n                    return\n        gc.collect()\n    mark("%s ALL %d IMAGES FINISHED in %.0fs" % (tag, len(my_paths), time.time() - t_start))\n\n\n# ------------------------------------------------------------------ orchestration\n\ndef collect_scans():\n    unzip_dir = os.path.join(pages_dir(), "input_pages")\n    os.makedirs(unzip_dir, exist_ok=True)\n    if os.path.exists(input_root()):\n        for root, dirs, files in os.walk(input_root()):\n            for name in files:\n                if not name.endswith(".mcsenc"):\n                    continue\n                import mcs_crypto\n                if _KEY["value"] is None:\n                    raise RuntimeError("Found encrypted scans (%s) but encryption is off for this run." % name)\n                plain = os.path.join(pages_dir(), "scans.zip")\n                mcs_crypto.decrypt_file(os.path.join(root, name), plain, _KEY["value"])\n                with zipfile.ZipFile(plain) as z:\n                    z.extractall(unzip_dir)\n                os.remove(plain)\n                print("[Decrypt] Decrypted and extracted \'%s\'" % name, flush=True)\n            for zf in files:\n                if zf.lower().endswith(".zip") and "model" not in zf.lower():\n                    try:\n                        with zipfile.ZipFile(os.path.join(root, zf)) as z:\n                            z.extractall(unzip_dir)\n                        print("[Auto-Unzip] Extracted \'%s\'" % zf, flush=True)\n                    except Exception as ze:\n                        print("[Auto-Unzip Warning] %s: %s" % (zf, ze), flush=True)\n\n    paths = []\n    for root, dirs, files in os.walk(unzip_dir):\n        paths += [os.path.join(root, f) for f in files if f.lower().endswith(IMAGE_EXTS) and not f.startswith(".")]\n    if not paths and os.path.exists(input_root()):\n        for root, dirs, files in os.walk(input_root()):\n            if "model" in root.lower() or "lora" in root.lower():\n                continue\n            paths += [os.path.join(root, f) for f in files if f.lower().endswith(IMAGE_EXTS) and not f.startswith(".")]\n    if not paths:\n        for root, dirs, files in os.walk(input_root()):\n            print("  %s -> %s" % (root, files), flush=True)\n        raise FileNotFoundError("No manga scans found in %s" % input_root())\n    return sorted(paths)\n\n\ndef split_spreads(paths):\n    """TPU only: splits double-page spreads into two portrait halves (one canvas shape)."""\n    if not fixed_canvas():\n        return list(paths), {}\n    split_dir = os.path.join(pages_dir(), "split_pages")\n    os.makedirs(split_dir, exist_ok=True)\n    queue, spreads = [], {}\n    for p in paths:\n        try:\n            with Image.open(p) as img:\n                w, h = img.size\n                if w <= h * 1.15:\n                    queue.append(p)\n                    continue\n                stem = os.path.splitext(os.path.basename(p))[0]\n                halves = []\n                for side, box in (("left", (0, 0, w // 2, h)), ("right", (w // 2, 0, w, h))):\n                    half = os.path.join(split_dir, "%s__spread_%s.png" % (stem, side))\n                    img.crop(box).save(half)\n                    halves.append(half)\n                queue += halves\n                spreads[stem] = (w, h)\n                print("[Auto-Split] \'%s\' (%dx%d) split into two halves." % (os.path.basename(p), w, h), flush=True)\n        except Exception as e:\n            print("Error inspecting \'%s\': %s" % (p, e), flush=True)\n            queue.append(p)\n    return queue, spreads\n\n\ndef stitch_spreads(spreads):\n    out_dir = os.path.join(pages_dir(), "generated")\n    for stem, (orig_w, orig_h) in spreads.items():\n        left = os.path.join(out_dir, "edit_%s__spread_left.png" % stem)\n        right = os.path.join(out_dir, "edit_%s__spread_right.png" % stem)\n        if not (os.path.exists(left) and os.path.exists(right)):\n            print("[Auto-Stitch Warning] Missing halves for \'%s\'" % stem, flush=True)\n            continue\n        l_img, r_img = Image.open(left).convert("RGB"), Image.open(right).convert("RGB")\n        h = max(l_img.height, r_img.height)\n        l_img = l_img.resize((int(l_img.width * h / l_img.height), h), Image.LANCZOS)\n        r_img = r_img.resize((int(r_img.width * h / r_img.height), h), Image.LANCZOS)\n        stitched = Image.new("RGB", (l_img.width + r_img.width, h))\n        stitched.paste(l_img, (0, 0))\n        stitched.paste(r_img, (l_img.width, 0))\n        if stitched.size != (orig_w, orig_h):\n            stitched = stitched.resize((orig_w, orig_h), Image.LANCZOS)\n        stitched.save(os.path.join(out_dir, "edit_%s.png" % stem))\n        os.remove(left)\n        os.remove(right)\n        print("[Auto-Stitch] Stitched \'edit_%s.png\' (%dx%d)" % (stem, orig_w, orig_h), flush=True)\n\n\ndef package_outputs():\n    """Zips the colorized pages into one archive in /kaggle/working (encrypted when encryption\n    is on) and removes every scratch file. Returns the archive\'s file name."""\n    work, pages = work_dir(), pages_dir()\n    out_dir = os.path.join(pages, "generated")\n    archive = "generated.zip"\n    if os.path.isdir(out_dir):\n        plain = os.path.join(pages, "generated.zip")\n        with zipfile.ZipFile(plain, "w", compression=zipfile.ZIP_DEFLATED) as zf:\n            for f in sorted(os.listdir(out_dir)):\n                zf.write(os.path.join(out_dir, f), arcname="generated/" + f)\n        shutil.rmtree(out_dir, ignore_errors=True)\n        if encrypted():\n            import mcs_crypto\n            archive = "generated" + mcs_crypto.EXTENSION\n            mcs_crypto.encrypt_file(plain, os.path.join(work, archive), _KEY["value"])\n            os.remove(plain)\n        elif plain != os.path.join(work, archive):\n            shutil.move(plain, os.path.join(work, archive))\n    for base in {work, pages}:\n        for name in ("split_pages", "input_pages", "loras", "colorize_worker.py", "mcs_crypto.py"):\n            p = os.path.join(base, name)\n            if os.path.isdir(p):\n                shutil.rmtree(p, ignore_errors=True)\n            elif os.path.isfile(p):\n                os.remove(p)\n    return archive\n\n\ndef spawn_workers(queue):\n    n = world_size()\n    if backend() != "tpu":\n        n = max(1, min(n, len(queue)))  # an idle GPU process would still load the whole model\n    if backend() == "tpu":\n        import torch_xla.distributed.xla_multiprocessing as xmp\n        xmp.spawn(mp_worker, args=(queue, n), nprocs=None)\n    elif n > 1:\n        import torch.multiprocessing as mp\n        mp.spawn(mp_worker, args=(queue, n), nprocs=n, join=True)\n    else:\n        mp_worker(0, queue, 1)\n    return n\n\n\ndef main():\n    hf_token = os.getenv("HF_TOKEN") or secret("HF_TOKEN")\n    if hf_token:\n        os.environ["HF_TOKEN"] = hf_token\n    os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "120")\n    os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "120")\n    os.makedirs(work_dir(), exist_ok=True)\n    os.makedirs(pages_dir(), exist_ok=True)\n    mark("Backend: %s | model: %s | %d LoRA(s)%s" % (backend(), _model_repo(), len(cfg("loras", [])),\n                                                     " | encrypted" if encrypted() else ""))\n    if encrypted():\n        load_key()  # fail in seconds, before any model download, if the secret is missing or wrong\n        mark("Encryption key verified (fingerprint %s)." % cfg("key_fingerprint"))\n\n    if prepare_local_model():\n        mark("Model found in mounted dataset; no download needed.")\n    else:\n        mark("No cached model mounted; downloading %s from Hugging Face..." % _model_repo())\n        warm_model_cache()\n\n    queue, spreads = split_spreads(collect_scans())\n    mark("Work queue: %d pages (%d spreads split)" % (len(queue), len(spreads)))\n\n    # Resolve every LoRA once, so the worker processes don\'t all download the same file.\n    for lora in cfg("loras", []):\n        print("[LoRA] %s -> %s" % (lora["path"], resolve_lora_path(lora["path"])), flush=True)\n\n    t_start = time.time()\n    try:\n        n = spawn_workers(queue)\n        mark("ALL %d WORKERS COMPLETED in %.0fs" % (n, time.time() - t_start))\n    except Exception as e:\n        mark("[Warning] Worker run ended with: %r" % e)\n    stitch_spreads(spreads)\n    archive = package_outputs()\n    mark("Outputs packaged into %s" % archive)\n'
CRYPTO_SOURCE = '"""Passphrase encryption for scans and results stored on Kaggle.\n\nUsed by the app (encrypt scans before upload, decrypt results after download) and by the\nKaggle notebook (decrypt scans, encrypt results). Needs only the `cryptography` package.\n\nFile format (all integers big-endian):\n    MAGIC (8) | salt (16) | nonce prefix (8) | chunks...\n    chunk = ciphertext length (4) | AES-256-GCM ciphertext + tag\n\nThe key is scrypt(passphrase, salt). Chunk i uses nonce = prefix + i (4 bytes) and the header,\nthe index and a "last chunk" flag as associated data, so a wrong key, tampering, reordering\nor truncation all fail loudly instead of producing garbage.\n"""\nimport hashlib\nimport os\nimport struct\nfrom pathlib import Path\n\nMAGIC = b"MCSENC1\\n"\nCHUNK = 4 * 1024 * 1024\nEXTENSION = ".mcsenc"\n# Name of the key file inside the temporary per-run key dataset on Kaggle.\nKEY_FILE = "mcs-key.txt"\n\n\nclass DecryptError(Exception):\n    pass\n\n\ndef fingerprint(passphrase: str) -> str:\n    """Short, non-secret identifier of a key: lets the notebook check it has the same key\n    as the app without either side revealing it."""\n    return hashlib.sha256(b"mcs-fingerprint:" + passphrase.encode("utf-8")).hexdigest()[:12]\n\n\ndef _derive(passphrase: str, salt: bytes) -> bytes:\n    return hashlib.scrypt(passphrase.encode("utf-8"), salt=salt, n=2 ** 15, r=8, p=1, maxmem=64 * 1024 * 1024, dklen=32)\n\n\ndef _aad(header: bytes, index: int, last: bool) -> bytes:\n    return header + struct.pack(">I?", index, last)\n\n\ndef encrypt_file(src, dst, passphrase: str):\n    from cryptography.hazmat.primitives.ciphers.aead import AESGCM\n\n    if not passphrase:\n        raise ValueError("An encryption key is required.")\n    salt, prefix = os.urandom(16), os.urandom(8)\n    header = MAGIC + salt + prefix\n    aes = AESGCM(_derive(passphrase, salt))\n    size = Path(src).stat().st_size\n    with open(src, "rb") as fin, open(dst, "wb") as fout:\n        fout.write(header)\n        index, done = 0, 0\n        while True:\n            block = fin.read(CHUNK)\n            done += len(block)\n            last = done >= size\n            ct = aes.encrypt(prefix + struct.pack(">I", index), block, _aad(header, index, last))\n            fout.write(struct.pack(">I", len(ct)) + ct)\n            index += 1\n            if last:\n                break\n\n\ndef decrypt_file(src, dst, passphrase: str):\n    from cryptography.exceptions import InvalidTag\n    from cryptography.hazmat.primitives.ciphers.aead import AESGCM\n\n    size = Path(src).stat().st_size\n    with open(src, "rb") as fin:\n        header = fin.read(len(MAGIC) + 24)\n        if len(header) < len(MAGIC) + 24 or not header.startswith(MAGIC):\n            raise DecryptError("Not an encrypted Manga Color archive.")\n        salt, prefix = header[len(MAGIC):len(MAGIC) + 16], header[len(MAGIC) + 16:]\n        aes = AESGCM(_derive(passphrase, salt))\n        tmp = Path(str(dst) + ".part")\n        try:\n            with open(tmp, "wb") as fout:\n                index = 0\n                while True:\n                    raw_len = fin.read(4)\n                    if len(raw_len) < 4:\n                        raise DecryptError("The encrypted archive is truncated.")\n                    (n,) = struct.unpack(">I", raw_len)\n                    ct = fin.read(n)\n                    nonce = prefix + struct.pack(">I", index)\n                    last = fin.tell() >= size\n                    try:\n                        block = aes.decrypt(nonce, ct, _aad(header, index, last))\n                    except InvalidTag:\n                        if index == 0:\n                            raise DecryptError("Wrong encryption key (or the archive is damaged).")\n                        raise DecryptError("The encrypted archive is damaged or was tampered with.")\n                    fout.write(block)\n                    index += 1\n                    if last:\n                        break\n            tmp.replace(dst)\n        finally:\n            if tmp.exists():\n                tmp.unlink()\n'
os.makedirs('/tmp/mcs_code', exist_ok=True)
for name, src in (('colorize_worker.py', WORKER_SOURCE), ('mcs_crypto.py', CRYPTO_SOURCE)):
    with open(os.path.join('/tmp/mcs_code', name), 'w', encoding='utf-8') as f:
        f.write(src)
if '/tmp/mcs_code' not in sys.path:
    sys.path.insert(0, '/tmp/mcs_code')
import colorize_worker
colorize_worker.main()
